[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haricalzi/Scientific-visualization/blob/main/Project1_Clustering/lect_data_vis_DIGA.ipynb)

# Laboratorio di data visualization: prevenzione cardiaca con l'intelligenza artificiale

Benvenuti in questo laboratorio! Oggi vestiremo i panni di un **data scientist** in ambito medico. 

Abbiamo a disposizione i dati clinici di **299 pazienti** che hanno sofferto di insufficienza cardiaca.

### Obiettivo
Vogliamo capire se i dati possono aiutarci a raggruppare i pazienti in base alla gravità della loro situazione ed estrarre potenziali informazioni utili dai pattern che troveremo.

---

Iniziamo!

In [ ]:
# installazione delle librerie necessarie
!pip install ucimlrepo matplotlib scikit-learn seaborn numpy pandas

In [ ]:
from ucimlrepo import fetch_ucirepo 

# Messaggio per confermare l'inizio dell'operazione
print("Scaricamento del dataset in corso...")

# Scarichiamo il dataset originale tramite il suo ID univoco (519)
heart_failure_clinical_records = fetch_ucirepo(id=519) 

print("Dataset scaricato con successo!")

In [ ]:
# Organizziamo i dati in tabelle chiamate DataFrame
# X contiene le caratteristiche cliniche (età, pressione, ecc.)
X = heart_failure_clinical_records.data.features 

# y contiene l'etichetta di sopravvivenza (il risultato che vorremmo prevedere)
y = heart_failure_clinical_records.data.targets 

print("Dati estratti e pronti per l'analisi.")

In [ ]:
# Visualizziamo le informazioni generali sul dataset per capire da dove viene
print("--- Informazioni Generali (Metadata) ---")
print(heart_failure_clinical_records.metadata) 

In [ ]:
# Visualizziamo la descrizione di ogni singola colonna
print("\n--- Dettaglio delle Variabili ---")
print(heart_failure_clinical_records.variables)

In [ ]:
# Mostriamo le prime 5 righe della tabella X per vedere i valori numerici
print("Anteprima dei dati dei primi 5 pazienti:")
X.head()

### Perché automatizzare la ricerca?

Se provate a guardare la tabella che abbiamo stampato sopra, vedrete centinaia di numeri. Per un essere umano è quasi impossibile "estrarre" informazioni utili o capire quali pazienti si somigliano semplicemente scorrendo queste righe. 

È come cercare degli schemi in un mare di cifre: l'occhio umano si stanca e si confonde. Per questo motivo, utilizzeremo delle **tecniche automatizzate** che permettono al computer di analizzare migliaia di dati in pochi istanti e trovare dei collegamenti nascosti che noi non riusciremmo mai a vedere.

---

In [ ]:
# ISTOGRAMMA DELL'ETÀ DEI PAZIENTI

# Importiamo le librerie per disegnare i grafici
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Uniamo temporaneamente le caratteristiche (X) e il risultato (y) in un'unica tabella 
# Questo ci renderà molto più facile creare grafici colorati!
df = pd.concat([X, y], axis=1)

# Impostiamo uno stile pulito per i grafici (rimuove sfondi grigi e linee inutili)
sns.set_theme(style="whitegrid")

# Creiamo la figura (il "foglio" su cui disegniamo)
plt.figure(figsize=(10, 6))

# Disegniamo l'istogramma dell'età
sns.histplot(data=df, x='age', bins=15, color='#4C72B0', kde=True, alpha=0.6)

# Aggiungiamo un titolo e le etichette degli assi per rendere il grafico più chiaro
plt.title('Distribuzione dell\'Età dei Pazienti', fontsize=16, pad=15)
plt.xlabel('Età (anni)', fontsize=14)
plt.ylabel('Numero di Pazienti', fontsize=14)

# Mostriamo il grafico
plt.show()

In [ ]:
# ISTOGRAMMA DELL'ESITO CLINICO DEI PAZIENTI

plt.figure(figsize=(8, 6))
palette_esito = {'Sopravvissuti': '#55A868', 'Deceduti': '#C44E52'}

# Nel dataset, 0 significa Sopravvissuto e 1 significa Deceduto.
df_plot = df.copy()
df_plot['death_event_label'] = df_plot['death_event'].map({0: 'Sopravvissuti', 1: 'Deceduti'})

# Creiamo il grafico a barre con colori ben distinti
ax = sns.countplot(
    data=df_plot,
    x='death_event_label',
    palette=palette_esito,
    hue='death_event_label',
    alpha=0.6
    )

plt.title('Esito Clinico dei Pazienti', fontsize=16, pad=15)
plt.xlabel('', fontsize=14) # Lasciamo vuoto, le etichette dicono già tutto
plt.ylabel('Numero di Pazienti', fontsize=14)

# Aggiungiamo le etichette sopra ogni barra per mostrare i numeri esatti
for container in ax.containers:
    ax.bar_label(container, fontsize=12, padding=3)

plt.show()

In [ ]:
# BOXPLOT DELLA FRAZIONE DI EIEZIONE PER SOPRAVVISSUTI E DECEDUTI

# La frazione di eiezione è una misura della salute del cuore: più è alta, meglio è (il cuore pompa più sangue ad ogni battito)

plt.figure(figsize=(9, 6))
sns.boxplot(
    data=df_plot, 
    x='death_event_label', 
    y='ejection_fraction', 
    hue='death_event_label',
    palette=palette_esito,
    legend=False,
    width=0.5,
    boxprops={'alpha': 0.7}
)

plt.title('Efficienza del Cuore: Sopravvissuti vs Deceduti', fontsize=16, pad=15)
plt.xlabel('', fontsize=14)
plt.ylabel('Frazione di Eiezione (%)', fontsize=14)

plt.show()

### Guida alla lettura del boxplot

Il **Boxplot** è uno degli strumenti più potenti per vedere come sono distribuiti i dati. Ecco come interpretare ogni sua parte:

1. **La scatola (box):** rappresenta il cuore dei dati, ovvero il **50% centrale** del campione.
   * Il bordo inferiore è il **25° percentile** (Q1): il 25% dei pazienti ha un valore inferiore a questo.
   * Il bordo superiore è il **75° percentile** (Q3): il 75% dei pazienti ha un valore inferiore a questo.
2. **La linea centrale:** rappresenta la **mediana** (50° percentile). È il valore che divide esattamente a metà i pazienti. Se la linea non è al centro della scatola, i dati sono sbilanciati (asimmetrici).
3. **I baffi (whiskers):** le linee che si estendono sopra e sotto. Indicano l'intervallo in cui si trovano i dati "normali" (solitamente fino a 1.5 volte l'ampiezza della scatola).
4. **I puntini (outliers):** se vedi dei punti oltre i baffi, sono i cosiddetti "fuori quota". Sono pazienti con valori molto rari o anomali rispetto al resto del gruppo.
5. **Notch (rientranza):** aiuta a confrontare le mediane: se le rientranze di due gruppi non si sovrappongono, è molto probabile che la differenza tra i due gruppi sia statisticamente significativa.

In [ ]:
# SCATTER PLOT: RELAZIONE TRA SALUTE DEL CUORE E DEI RENI

# La frazione di eiezione è una misura della salute del cuore: più è alta, meglio è (il cuore pompa più sangue ad ogni battito)

# La creatinina sierica è una misura della salute dei reni: più è alta, peggio è (i reni non riescono a filtrare le tossine)

plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=df_plot, 
    x='ejection_fraction', 
    y='serum_creatinine', 
    hue='death_event_label', 
    palette=palette_esito,
    s=80,
    alpha=0.7
)

plt.title('Cuore vs Reni: Relazione con l\'Esito Clinico', fontsize=16, pad=15)
plt.xlabel('Frazione di Eiezione (%, Efficienza del Cuore)', fontsize=14)
plt.ylabel('Creatinina Sierica (mg/dL, Tossine nei Reni)', fontsize=14)

plt.legend(title='Esito', fontsize=12, title_fontsize=12)

plt.show()

---

### Cos'è il clustering?

Il **clustering** è una tecnica di **apprendimento non supervisionato** (*unsupervised learning*). In questo tipo di analisi non diamo al computer delle "risposte giuste" (come sapere già chi è malato e chi no), ma gli chiediamo di osservare i dati e raggruppare i pazienti che si somigliano di più tra loro.



**Esempio pratico:** immaginate di dover organizzare una libreria di migliaia di canzoni senza conoscere i generi musicali. Il computer analizzerà il ritmo, gli strumenti e la velocità per creare dei gruppi (cluster) di brani simili. Alla fine potreste trovarvi con un gruppo "rock" e uno "jazz" anche senza avergli mai spiegato cosa siano!

---

### Come decidiamo il numero di gruppi? ($k$)

Alcuni algoritmi, come il **K-Means**, richiedono che sia il ricercatore a decidere in anticipo in quanti gruppi (chiamati **$k$**) dividere i dati. Scegliere un numero a caso non è professionale, quindi utilizziamo due tecniche matematiche per trovare il valore ottimale:

1. **Metodo del Gomito (Elbow Method):**
   * **Descrizione tecnica:** si basa sul calcolo dell'**Inerzia** (o WCSS), ovvero la somma delle distanze al quadrato tra ogni punto e il centro del suo gruppo. 
   * **Cosa cerchiamo:** più l'inerzia è bassa, più i gruppi sono "compatti". Nel grafico, cerchiamo il punto in cui la curva "piega" bruscamente (come un gomito): dopo quel punto, aggiungere altri gruppi riduce l'inerzia di pochissimo, rendendo il modello inutilmente complicato.

2. **Metodo della Silhouette:**
   * **Descrizione tecnica:** questo metodo misura quanto un punto è simile al proprio gruppo rispetto agli altri.
   * **Cosa cerchiamo:** un valore vicino a **1** significa che i gruppi sono molto densi e ben separati tra loro. Un valore vicino allo **0** indica che i gruppi si sovrappongono, mentre valori negativi indicano che i punti sono stati assegnati al gruppo sbagliato.

### In sintesi: che differenza c'è?

Per capire meglio, possiamo immaginare i cluster come dei gruppi di amici in una stanza:

* **Elbow (Inerzia) -> Guarda "dentro":** misura quanto ogni persona è vicina al centro del proprio gruppo. Si concentra solo sulla **Coesione**. Se il gruppo è molto unito e tutti sono vicini al centro, l'inerzia è bassa.


* **Silhouette -> Guarda "dentro e fuori":** è un test più severo. Non si chiede solo "quanto sei vicino ai tuoi amici?", ma anche "quanto sei lontano dagli altri gruppi?". Misura quindi sia la **Coesione** (dentro) che la **Separazione** (fuori). 


**Il verdetto:** un "buon" gruppo per la Silhouette è un gruppo dove i membri si stanno simpatici tra loro (vicini) ma, allo stesso tempo, non sopportano gli altri gruppi (lontani).

In [ ]:
# Elbow Method per trovare il numero ottimale di gruppi (cluster)
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# WCSS (Within Cluster Sum of Squares)
# Rappresenta l'inerzia: cerchiamo il valore di k dove l'inerzia smette di scendere velocemente e la curva crea una sorta di "gomito"
wcss = []

# Proviamo a dividere i dati in un numero di gruppi che va da 1 a 10
k_range = range(1, 11)

# Calcoliamo l'inerzia per ogni possibile numero di gruppi
for k in k_range:
    # Inizializziamo l'algoritmo K-Means
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto', max_iter=300)
    # Facciamo analizzare i dati al computer
    kmeans.fit(X)
    # Salviamo il valore dell'inerzia nella nostra lista
    wcss.append(kmeans.inertia_)

# Creazione del grafico del "Gomito"
plt.figure(figsize=(10, 6))
plt.plot(k_range, wcss, marker='o', color='blue', linestyle='--')
plt.title('Metodo del Gomito (Elbow Method)')
plt.xlabel('Numero di gruppi (k)')
plt.ylabel('Inerzia (WCSS)')
plt.xticks(k_range)
plt.grid(True)
plt.show()


In [ ]:
# Analisi del Punteggio Silhouette per convalidare il numero ideale di gruppi
from sklearn.metrics import silhouette_score

# Creiamo una lista per memorizzare i punteggi di ogni prova
silhouette_scores = []

# Proviamo diversi valori di k (numero di gruppi)
# Nota: partiamo da 2 perché non si può calcolare la silhouette per un gruppo solo!
k_range = range(2, 11)

# Calcoliamo il punteggio per ogni k
for k in k_range:
    # Inizializziamo l'algoritmo K-Means
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto', max_iter=300)
    
    # Assegniamo ogni paziente a un gruppo (labels)
    cluster_labels = kmeans.fit_predict(X)
    
    # Calcoliamo quanto è "buona" questa divisione
    # Più il punteggio è alto, meglio sono definiti i gruppi
    score = silhouette_score(X, cluster_labels)
    
    # Salviamo il risultato
    silhouette_scores.append(score)
    #print(f'Per k = {k}, il punteggio silhouette è {score}')

# Creazione del grafico della Silhouette
plt.figure(figsize=(10, 6))
plt.plot(k_range, silhouette_scores, marker='o', color='green', linestyle='--')
plt.title('Punteggio Silhouette per trovare il k ottimale')
plt.xlabel('Numero di gruppi (k)')
plt.ylabel('Punteggio Silhouette')
plt.xticks(k_range)
plt.grid(True)
plt.show()

### Come ragiona l'algoritmo K-Means?

Il **K-Means** è uno degli algoritmi più famosi per fare clustering. Il suo obiettivo è dividere i dati in un numero $k$ di gruppi (che abbiamo scelto noi grazie ai grafici precedenti). 

Ma come fa "fisicamente" a creare questi gruppi? Segue un processo iterativo (a ripetizione) in 3 passi:

1.  **Inizializzazione:** l'algoritmo piazza a caso nel grafico $k$ punti speciali chiamati **Centroidi** (i "centri" dei futuri gruppi).
2.  **Assegnazione:** ogni paziente viene assegnato al centroide più vicino. Per misurare la "vicinanza", l'algoritmo usa la **Distanza Euclidea** (quella che si calcola con il Teorema di Pitagora).
3.  **Aggiornamento:** l'algoritmo sposta ogni centroide esattamente nel "centro matematico" (la media) di tutti i pazienti che gli sono stati assegnati.

**Il ciclo si ripete:** l'algoritmo ri-assegna i pazienti ai nuovi centri e sposta di nuovo i centri, finché i gruppi non diventano stabili e non cambiano più. 

![Animazione K-Means](https://upload.wikimedia.org/wikipedia/commons/e/ea/K-means_convergence.gif)

### La Distanza Euclidea
Per due punti nel piano $P_1(x_1, y_1)$ e $P_2(x_2, y_2)$, la distanza è:
$$d = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}$$
Più piccola è questa distanza, più il paziente è simile al "paziente tipo" (il centroide) di quel gruppo.

---

### Come facciamo a vedere i dati in 12 dimensioni?

Il nostro dataset ha 12 caratteristiche (età, pressione, ecc.). Noi esseri umani riusciamo a visualizzare graficamente solo 2 o 3 dimensioni (lunghezza, larghezza e altezza). Come facciamo a vedere i gruppi se i dati si muovono in uno spazio a 12 dimensioni?

Usiamo la **PCA (Principal Component Analysis)**, una tecnica che "schiaccia" le 12 dimensioni in 2 sole dimensioni artificiali (chiamate componenti), cercando di perdere meno informazioni possibili.

---

In [ ]:
# Applicazione di k-means con k=2 e grafico dei cluster dopo riduzione a 2D con PCA

from sklearn.decomposition import PCA

# Riduciamo a 2 componenti per poter creare il grafico
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X.values)  

# Eseguiamo il K-Means sui dati originali (X)
kmeans = KMeans(n_clusters=2, random_state=42, n_init='auto', max_iter=300)
y_kmeans = kmeans.fit_predict(X)

# Creazione del grafico dei cluster
plt.figure(figsize=(10, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_kmeans, s=50, cmap='RdYlBu', alpha=0.6)

# Calcoliamo e proiettiamo i centri dei gruppi
centers = kmeans.cluster_centers_
centers_pca = pca.transform(centers)

# Disegniamo i centroidi come 'X' rosse
plt.scatter(centers_pca[:, 0], centers_pca[:, 1], c='red', s=200, alpha=0.75, marker='X')

plt.title('K-Means Clustering con k=2')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.grid(True)
plt.show()

### Problema

I gruppi sembrano tagliati e schiacciati in modo strano

In [ ]:
# Grafico dell'intervallo di valori (range) delle caratteristiche con un grafico a barre

# Calcoliamo l'intervallo per ogni caratteristica (Valore Massimo - Valore Minimo)
intervallo_caratteristiche = X.max() - X.min()

nomi_caratteristiche = intervallo_caratteristiche.index.tolist()
valori_intervallo = intervallo_caratteristiche.values

# Creazione del grafico a barre orizzontali
plt.figure(figsize=(10, 8))
plt.barh(nomi_caratteristiche, valori_intervallo, color='skyblue', edgecolor='navy', alpha=0.6) 

plt.title('Intervallo di valori per ogni caratteristica (Range)')
plt.xlabel('Range (Massimo - Minimo)')
plt.ylabel('Caratteristiche (Features)')

# Invertiamo l'asse Y per leggere le caratteristiche dall'alto verso il basso
plt.gca().invert_yaxis() 

# Aggiungiamo una griglia per rendere i numeri più leggibili
plt.grid(axis='x', linestyle='--', alpha=0.7)

# Ottimizziamo lo spazio per non tagliare le scritte
plt.tight_layout() 

plt.show()


<details>
<summary><b>⚠️ Il problema: perché i numeri grandi possono ingannare l'IA?</b></summary>

### Variabili con scale diverse

Nel nostro dataset abbiamo un mix di dati:
* **Variabili continue:** numeri grandi (es. Piastrine fino a 800.000).
* **Variabili booleane:** solo 0 o 1 (es. Fumatore Sì/No).

**Perché è un problema?**
L'algoritmo è "ingannato" dai numeri grandi: penserebbe che le piastrine siano migliaia di volte più importanti del fumo solo perché i valori sono più alti. Questo sbilancerebbe totalmente il calcolo delle distanze nel clustering.
</details>

---


## La soluzione: standardizzazione (Scaling)

Per riportare ogni caratteristica sulla stessa scala, applichiamo a ogni dato questa formula matematica:

$$z = \frac{x - \mu}{\sigma}$$

**Dove:**
* **$x$**: è il valore originale (es. 70 anni).
* **$\mu$ (mu)**: è la **media** di quella colonna.
* **$\sigma$ (sigma)**: è la **deviazione standard** (indica quanto i dati sono vari o "sparpagliati").
* **$z$**: è il valore finale "scalato".

**Cosa abbiamo ottenuto?** Dopo questa trasformazione, il valore **0** rappresenta esattamente la media. Un valore positivo significa "sopra la media", uno negativo "sotto la media". Ora il computer può confrontare l'età con i livelli di piastrine senza fare confusione!

In [ ]:
# Scaliamo le caratteristiche prima di fare il clustering usando StandardScaler
from sklearn.preprocessing import StandardScaler

# Recuperiamo i dati originali (ne teniamo una copia per sicurezza)
heart_failure_clinical_records = fetch_ucirepo(id=519) 
X_original = heart_failure_clinical_records.data.features 
y = heart_failure_clinical_records.data.targets 

# Inizializziamo lo scaler (lo strumento che applica la formula dello Z-score)
scaler = StandardScaler()

# Applichiamo lo scaling ai dati: ora ogni colonna peserà allo stesso modo
X_scaled = scaler.fit_transform(X_original)

print("Scaling applicato con successo ai dati")

In [ ]:
import pandas as pd

# Trasformiamo i dati scalati in una tabella leggibile con i nomi delle colonne originali
X_scaled_df = pd.DataFrame(X_scaled, columns=X_original.columns)

# Stampiamo le prime 5 righe per vedere la differenza
X_scaled_df.head()

In [ ]:
# Elbow Method sui dati SCALATI

# Lista per memorizzare i valori di WCSS (Inerzia)
wcss = []
k_range = range(1, 11)

# Calcoliamo l'inerzia per ogni k provato (da 1 a 10)
for k in k_range:
    # Usiamo i dati scalati (X_scaled) per un calcolo corretto delle distanze
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    kmeans.fit(X_scaled) 
    wcss.append(kmeans.inertia_)

# Creazione del grafico
plt.figure(figsize=(10, 6))
plt.plot(k_range, wcss, marker='o', color='blue', linestyle='--')
plt.title('Metodo del Gomito (su dati scalati)')
plt.xlabel('Numero di gruppi (k)')
plt.ylabel('Inerzia (WCSS)')
plt.xticks(k_range)
plt.grid(True)
plt.show()

In [ ]:
# Analisi della Silhouette sui dati SCALATI per convalidare il numero di gruppi

silhouette_scores = []
k_range = range(2, 11) 

# Calcoliamo il punteggio di silhouette per ogni k (numero di gruppi)
for k in k_range:
    # Usiamo i dati scalati per una valutazione corretta della separazione
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    cluster_labels = kmeans.fit_predict(X_scaled)
    
    # Il punteggio misura quanto i cluster sono compatti e ben separati
    score = silhouette_score(X_scaled, cluster_labels)
    silhouette_scores.append(score)
    # print(f'Per k = {k}, il punteggio Silhouette è: {score:.4f}')

# Creazione del grafico della Silhouette
plt.figure(figsize=(10, 6))
plt.plot(k_range, silhouette_scores, marker='o', color='green', linestyle='--')
plt.title('Punteggio Silhouette (su dati scalati)')
plt.xlabel('Numero di gruppi (k)')
plt.ylabel('Punteggio Silhouette')
plt.xticks(k_range)
plt.grid(True)
plt.show()

In [ ]:
# Applicazione di k-means con il k ottimale sui dati SCALATI e grafico 2D con PCA
from sklearn.decomposition import PCA

# Scegliamo 2 come k ottimale (basandoci sui grafici di Gomito e Silhouette precedenti)
OPTIMAL_K = 2 

# Riduzione della dimensionalità (PCA) sui dati SCALATI
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Applichiamo K-Means sui dati SCALATI
kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init='auto')
y_kmeans = kmeans.fit_predict(X_scaled)

# Visualizzazione dei cluster
plt.figure(figsize=(10, 6))

# Disegniamo i punti (pazienti) colorandoli in base al gruppo assegnato
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_kmeans, s=50, cmap='RdYlBu', alpha=0.7)

# Posizioniamo i centroidi (i centri dei gruppi) nello spazio PCA
centers_scaled = kmeans.cluster_centers_
centers_pca = pca.transform(centers_scaled)

# Disegniamo i Centroidi come grosse 'X' rosse
plt.scatter(centers_pca[:, 0], centers_pca[:, 1], c='red', s=250, marker='X', label='Centroidi')

plt.title(f'Clustering K-Means (k={OPTIMAL_K}) visualizzato tramite PCA')
plt.xlabel('Componente Principale 1')
plt.ylabel('Componente Principale 2')
plt.legend()
plt.grid(True)
plt.show()

### Identikit dei gruppi: cosa li differenzia?

Ora che abbiamo diviso i pazienti in due gruppi, dobbiamo capire "chi" sono. Usiamo una **Heatmap** (mappa di calore) per visualizzare il profilo medio di ogni cluster.

Poiché stiamo usando i dati **scalati** (Z-score):
* **Valori positivi (rosso):** indicano che quella caratteristica è **sopra la media** globale.
* **Valori negativi (blu):** indicano che quella caratteristica è **sotto la media** globale.
* **Valori vicini a 0 (bianco/grigio):** indicano valori in linea con la media di tutti i pazienti.

Questo ci permette di vedere a colpo d'occhio quali fattori clinici definiscono un gruppo rispetto all'altro.

In [ ]:
# Grafico Heatmap dei profili dei cluster basato sui dati scalati
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Creiamo un DataFrame con i dati scalati e aggiungiamo la colonna del cluster
X_analisi_scaled = pd.DataFrame(X_scaled, columns=X_original.columns)
X_analisi_scaled['cluster'] = y_kmeans

# Calcoliamo la media di ogni caratteristica per ogni cluster
# Queste medie rappresentano i profili Z-score (distanza dalla media globale)
medie_scalate = X_analisi_scaled.groupby('cluster').mean()

# Generazione della Heatmap
plt.figure(figsize=(10, 10))
sns.heatmap(
    medie_scalate.T, # Trasponiamo la tabella per avere le variabili sulle righe
    annot=True,      # Mostriamo i numeri dentro le celle
    cmap='coolwarm', # Colore blu per valori bassi, rosso per valori alti
    fmt=".2f",
    cbar_kws={'label': 'Media del Cluster (Z-score relativo alla media globale)'}
)

plt.title('Mappa di Calore dei Profili dei Cluster (Dati Scalati)')
plt.ylabel('Variabili Cliniche')
plt.xlabel('Cluster')
plt.xticks(rotation=0)
plt.show()

<details>
<summary><b>🕵️‍♂️ Interpretazione dei risultati: chi sono i pazienti?</b></summary>

Dall'analisi della Heatmap precedente, possiamo tracciare un identikit chiaro dei due gruppi creati dal clustering:

* **Cluster 0 (il profilo maschile):** è caratterizzato principalmente da **uomini fumatori**. Nella mappa di calore, vediamo valori positivi (rossi) in corrispondenza delle variabili `sex` e `smoking`.
* **Cluster 1 (il profilo femminile):** è composto prevalentemente da **donne non fumatrici**. Qui i valori sono negativi (blu), indicando l'assenza di fumo e il genere opposto rispetto alla codifica del dataset.
</details>


---


<details>
<summary><b>🤔 Perché l'IA sceglie proprio queste variabili per creare i gruppi?</b></summary>

#### Osservazione tecnica: il peso delle variabili binarie
Notiamo che le **variabili binarie** (quelle con solo due stati: 0 o 1, come fumo, sesso, diabete) sembrano "comandare" la formazione dei gruppi. 

Questo accade spesso nel clustering: poiché queste variabili creano una separazione netta (o sei dentro o sei fuori), l'algoritmo le trova molto "comode" per dividere i dati in blocchi distinti. In questo caso, l'IA ha deciso che la differenza di genere e l'abitudine al fumo fossero i tratti più forti per distinguere i pazienti nel dataset.
</details>

---

### Riunire i risultati: torniamo ai dati reali

Anche se abbiamo usato i dati "scalati" per far funzionare bene l'algoritmo, ora vogliamo analizzare i pazienti usando i loro valori originali. 

Aggiungiamo una nuova colonna al nostro dataset originale chiamata `cluster`. In questo modo, potremo vedere facilmente per ogni paziente se l'IA lo ha inserito nel **Gruppo 0** o nel **Gruppo 1**.

In [ ]:
# Aggiungiamo le etichette dei cluster al dataframe originale X
# In questo modo possiamo studiare i valori clinici reali per ogni gruppo
X['cluster'] = y_kmeans

# Stampiamo un messaggio di conferma e le prime righe della tabella aggiornata
print("Etichette dei cluster aggiunte con successo al dataset originale.")
X.head()

### Analisi statistica: cosa dicono i numeri?

Per capire davvero chi fa parte dei due gruppi, analizziamo le statistiche medie. 

**Nota per la classe:**
1.  **Variabili Continue:** Osserviamo la media e la mediana per capire il "paziente tipo" del gruppo.
2.  **Variabili Binarie (0/1):** Qui c'è un trucco! La **media** rappresenta la percentuale. Se leggiamo `smoking: mean 0.85`, significa che l'**85%** dei pazienti in quel gruppo fuma.

In [ ]:
# Statistiche dettagliate per ogni cluster
# Usiamo i dati scalati per vedere come si posizionano rispetto alla media globale (0)
gruppi = X_analisi_scaled.groupby('cluster')

# Distinguiamo tra variabili continue e binarie per una lettura più chiara
for nome, gruppo in gruppi:
    print(f"\n" + "="*30)
    print(f" IDENTIKIT CLUSTER {nome} ")
    print("="*30)
    
    # Variabili continue: mostriamo media, deviazione standard e mediana
    print("\n[Variabili Continue - Profilo Z-score]")
    print(gruppo.agg({
            'age': ['mean', 'std', 'median'],
            'creatinine_phosphokinase': ['mean', 'std', 'median'],
            'ejection_fraction': ['mean', 'std', 'median'],
            'platelets': ['mean', 'std', 'median'],
            'serum_creatinine': ['mean', 'std', 'median'],
            'serum_sodium': ['mean', 'std', 'median'],
            'time': ['mean', 'std', 'median'],
        }))
    
    # Variabili binarie: la media indica la proporzione di "Sì" (1) nel gruppo
    print("\n[Variabili Binarie - Proporzioni e Conteggio]")
    print(gruppo.agg({
            'anaemia': ['mean', 'count'], 
            'diabetes': ['mean', 'count'],
            'high_blood_pressure': ['mean', 'count'],
            'sex': ['mean', 'count'],
            'smoking': ['mean', 'count'],
        }))

In [ ]:
# Differenziamo i tipi di variabili per la visualizzazione
import seaborn as sns
import matplotlib.pyplot as plt

# Variabili continue (da visualizzare con i Boxplot)
continuous_vars = [
    'age', 'creatinine_phosphokinase', 'ejection_fraction',
    'platelets', 'serum_creatinine', 'serum_sodium', 'time'
]

# Variabili binarie (da visualizzare con i grafici a barre delle proporzioni)
binary_vars = [
    'anaemia', 'diabetes', 'high_blood_pressure', 'sex', 'smoking'
]

### Guida alla lettura del boxplot

Il **Boxplot** è uno degli strumenti più potenti per vedere come sono distribuiti i dati. Ecco come interpretare ogni sua parte:

1. **La scatola (box):** rappresenta il cuore dei dati, ovvero il **50% centrale** del campione.
   * Il bordo inferiore è il **25° percentile** (Q1): il 25% dei pazienti ha un valore inferiore a questo.
   * Il bordo superiore è il **75° percentile** (Q3): il 75% dei pazienti ha un valore inferiore a questo.
2. **La linea centrale:** rappresenta la **mediana** (50° percentile). È il valore che divide esattamente a metà i pazienti. Se la linea non è al centro della scatola, i dati sono sbilanciati (asimmetrici).
3. **I baffi (whiskers):** le linee che si estendono sopra e sotto. Indicano l'intervallo in cui si trovano i dati "normali" (solitamente fino a 1.5 volte l'ampiezza della scatola).
4. **I puntini (outliers):** se vedi dei punti oltre i baffi, sono i cosiddetti "fuori quota". Sono pazienti con valori molto rari o anomali rispetto al resto del gruppo.
5. **Notch (rientranza):** aiuta a confrontare le mediane: se le rientranze di due gruppi non si sovrappongono, è molto probabile che la differenza tra i due gruppi sia statisticamente significativa.

In [ ]:
# Boxplot per le variabili continue
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(16, 12))

for i, col in enumerate(continuous_vars, 1):
    plt.subplot(3, 3, i)
    
    sns.boxplot(x='cluster', y=col, data=X, notch=True, palette='Set2', hue='cluster', legend=False, boxprops={'alpha': 0.7})
    
    plt.title(f'Distribuzione di: {col.replace("_", " ").capitalize()}')
    plt.xlabel('Gruppo (Cluster)')
    plt.ylabel('Valore Reale')
    plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

### Analizzare le variabili binarie (Sì/No)

Per variabili come il **fumo**, il **diabete** o il **genere**, non possiamo usare le medie nel senso tradizionale. Usiamo invece un **conteggio (count plot)**.

In questi grafici:
* **L'asse X** divide i pazienti nei due Cluster (0 e 1).
* **I colori (hue)** indicano la presenza (1) o l'assenza (0) di una caratteristica.
* **L'altezza delle barre** ci dice quanti pazienti appartengono a quella categoria.

Questo è il momento della verità: vedremo chiaramente se un gruppo è composto quasi interamente da uomini o se il fumo è concentrato solo in un cluster.

In [ ]:
# Count Plot per le variabili binarie organizzati in una griglia
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(16, 10))

for i, col in enumerate(binary_vars, 1):
    plt.subplot(2, 3, i)
    
    # Creiamo il grafico a barre contanto le occorrenze
    ax = sns.countplot(data=X, x='cluster', hue=col, palette='viridis', alpha=0.7)
    
    plt.title(f'Distribuzione di: {col.capitalize()}')
    plt.xlabel('Gruppo (Cluster)')
    plt.ylabel('Numero di Pazienti')
    
    if col == "sex":
        labels = ['Donne (0)', 'Uomini (1)']
    else:
        labels = ['No (0)', 'Sì (1)']
    
    # Recuperiamo la legenda e cambiamo le etichette
    plt.legend(title=col.capitalize(), labels=labels)

# Pulizia del layout
plt.tight_layout()
plt.show()

<details>
<summary><b>👀 Cosa osserviamo analizzando i gruppi?</b></summary>

* **Cluster 0:** esclusivamente uomini fumatori e non fumatori.
* **Cluster 1:** quasi esclusivamente donne non fumatrici.
</details>

<details>
<summary><b>🧩 Perché accade questo? (Il limite nascosto)</b></summary>

In questo specifico database (raccolto in un contesto clinico specifico), la categoria delle **donne fumatrici** è praticamente assente. 

Di conseguenza, l'algoritmo di Clustering ha "imparato" che il fumo è una caratteristica quasi esclusivamente maschile. L'IA non ha il "senso comune": non sa che nel mondo reale esistono molte donne che fumano, vede solo ciò che i dati le mostrano.
</details>

In [ ]:
# Filtriamo il dataset per trovare solo le donne che fumano
donne_fumatrici = X[(X['sex'] == 0) & (X['smoking'] == 1)]

# Contiamo quante sono rispetto al totale delle donne
totale_donne = len(X[X['sex'] == 0])
numero_fumatrici = len(donne_fumatrici)

print(f"Pazienti totali nel dataset: {len(X)}")
print(f"Totale donne: {totale_donne}")
print(f"Di cui fumatrici: {numero_fumatrici}")

print(f"\nPercentuale di fumatrici tra le donne: {(numero_fumatrici/totale_donne)*100:.2f}%")


### La lezione importante: "garbage in, garbage out"
L'intelligenza artificiale non ha il "senso comune": non sa che nel mondo reale esistono molte donne che fumano. Lei si limita a trovare pattern nei dati che le forniamo. 
Se i dati sono sbilanciati o riflettono bias culturali/geografici, i gruppi creati dall'IA saranno distorti. **Il modello non descrive la verità assoluta, ma solo lo specchio dei dati che ha ricevuto.**

---

### Quali variabili "comandano" i gruppi?

Dopo aver guardato i grafici, chiediamo alla matematica di confermarci quali sono le caratteristiche che hanno davvero guidato l'IA. Usiamo dei test per scoprire quali variabili hanno "comandato" la divisione dei pazienti e quali sono invece solo frutto del caso.

In [ ]:
import pandas as pd
from scipy import stats

# Dividiamo i dati originali nei due cluster identificati
cluster_0 = X[X['cluster'] == 0]
cluster_1 = X[X['cluster'] == 1]

alpha = 0.05 # Soglia di significatività standard
results_list = []

# 1. Test di Kruskal-Wallis per le variabili continue
for var in continuous_vars:
    stat_kw, p_value_kw = stats.kruskal(cluster_0[var], cluster_1[var])
    results_list.append({
        'Caratteristica': var,
        'Test': 'Kruskal-Wallis',
        'P-Value': p_value_kw,
        'Significativo': 'Sì' if p_value_kw < alpha else 'No'
    })

# 2. Test del Chi-quadrato per le variabili binarie
for var in binary_vars:
    contingency_table = pd.crosstab(X['cluster'], X[var])
    # Verifichiamo che la tabella sia corretta per il test (2x2)
    if contingency_table.shape == (2, 2):
        chi2, p_value_chi2, _, _ = stats.chi2_contingency(contingency_table)
    else:
        p_value_chi2 = 1.0

    results_list.append({
        'Caratteristica': var,
        'Test': 'Chi-squared',
        'P-Value': p_value_chi2,
        'Significativo': 'Sì' if p_value_chi2 < alpha else 'No'
    })

# Creiamo il DataFrame riassuntivo
results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values(by='P-Value')

# Funzione per rendere leggibili i p-value molto piccoli
def format_pvalue(p):
    if p < 0.001:
        return "< 0.001"
    else:
        return f"{p:.4f}"

results_df.drop(columns=['P-Value', 'Test'], inplace=True)
results_df

<details>
<summary><b>🔍 Tiriamo le somme: cosa ha guidato davvero l'algoritmo?</b></summary>

### Cosa abbiamo scoperto?
L'algoritmo ha "visto" nel **genere** e nel **fumo** le coordinate principali per orientarsi. 

Questo accade perché le variabili binarie (Sì/No, Uomo/Donna) creano confini molto netti che il K-Means riesce a catturare facilmente, mentre le variabili mediche continue (come le piastrine) sono state usate solo per rifinire ulteriormente i gruppi. In pratica, l'IA ha scelto la "strada più facile" per dividere i pazienti, anche se non era quella più utile dal punto di vista medico.
</details>

---

## Convalida clinica: i cluster riflettono il rischio di morte?

Dopo aver diviso i pazienti in base a genere e fumo (i fattori dominanti in questo dataset), dobbiamo chiederci: **questa divisione ha un valore medico?** Incrociamo i nostri cluster con la colonna `DEATH_EVENT` (l'evento del decesso) per vedere se uno dei due gruppi ha un tasso di mortalità più alto.

In [ ]:
# Analisi del tasso di mortalità (DEATH_EVENT) per ogni cluster

# Aggiungiamo la variabile reale del decesso al dataframe X (usando i target y)
# Nota: y contiene la colonna 'DEATH_EVENT'
X['DEATH_EVENT'] = y.values

# Calcoliamo le percentuali esatte per il commento
mortality_rates = X.groupby('cluster')['DEATH_EVENT'].mean() * 100

print(f"Tasso di mortalità Cluster 0: {mortality_rates[0]:.2f}%")
print(f"Tasso di mortalità Cluster 1: {mortality_rates[1]:.2f}%")

# Rimuoviamo la colonna 'DEATH_EVENT' da X per evitare confusione nelle analisi future
X.drop(columns=['DEATH_EVENT'], inplace=True)

---

## Il verdetto finale

Abbiamo chiesto all'IA di dividere i pazienti e lei lo ha fatto perfettamente. Ma ora guardiamo la realtà dei fatti...

<details>
<summary><b>📊 1. Vediamo i numeri: qual è il tasso di mortalità nei due gruppi?</b></summary>

* **Tasso Cluster 0 (Uomini Fumatori):** ~32.3%
* **Tasso Cluster 1 (Donne Non Fumatrici):** ~31.8%

**Sorpresa:** I due gruppi hanno praticamente lo stesso identico rischio di morte.
</details>

<br>

<details>
<summary><b>🎓 2. Cosa abbiamo imparato da questo risultato inaspettato?</b></summary>

1.  **Struttura ≠ Rischio:** L'algoritmo ha trovato una "struttura" (ha diviso uomini e donne), ma questa divisione **non ha alcuna importanza clinica** per la sopravvivenza.
2.  **Variabili "Rumore":** Il genere e il fumo sono stati scelti dall'IA solo perché erano facili da dividere (0 o 1), ma non sono i veri colpevoli della mortalità.
3.  **L'importanza dei dati medici:** Fattori come l'età, la creatinina o la frazione di eiezione contano molto di più, anche se sono più difficili da raggruppare a occhio nudo.
</details>

<br>

<details>
<summary><b>🏁 3. La morale della favola (La lezione del Data Scientist)</b></summary>

**Non tutti i pattern trovati dall'Intelligenza Artificiale sono utili.** In questo esperimento abbiamo trovato un pattern **sociologico** (uomini fumatori vs donne non fumatrici), ma abbiamo fallito nel trovare un pattern **medico** (chi è davvero più a rischio). 

*Ricorda: l'IA vede le differenze, ma siamo noi umani a dover decidere quali differenze contano davvero.*
</details>

---

### 1. Apprendimento Supervisionato
Passiamo dal clustering alla **Classificazione**. In questa fase l'IA è **supervisionata**: le forniamo i dati dei pazienti insieme alla risposta corretta (**DEATH_EVENT**). Il modello impara dai casi passati per prevedere il rischio di pazienti futuri.

![Supervised vs Unsupervised Learning](https://upload.wikimedia.org/wikipedia/commons/4/4d/Supervised_and_unsupervised_learning.png)

### 2. Decision Tree e Random Forest
* **Albero Decisionale:** un diagramma di flusso che pone domande ai dati (es: "Età > 70?") per arrivare a una conclusione. 

* **Random Forest:** una "foresta" composta da centinaia di alberi indipendenti. Ogni albero esprime un voto e la decisione finale viene presa a **maggioranza**. è molto più robusta e precisa di un albero singolo.

<img src="https://miro.medium.com/v2/resize:fit:1400/1*b3kM3WFJB6b92ZHp_J871Q.jpeg" style="width:38%;">

### 3. Obiettivo: Feature Importance
Usiamo la Random Forest per calcolare la **Feature Importance**. Questo ci permetterà di capire se per l'IA sono più determinanti i parametri clinici (cuore e reni) o i fattori sociali (sesso e fumo) emersi precedentemente con il clustering.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt

# Creiamo un albero decisionale semplice per capire quali caratteristiche sono più importanti per distinguere i due cluster

# Usiamo X (dati originali) e non X_scaled, così le domande saranno leggibili 
tree_simple = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_simple.fit(X, y)

# Visualizziamo l'albero
plt.figure(figsize=(20, 10))
nodi = plot_tree(tree_simple, 
                 feature_names=X.columns, 
                 class_names=['Sopravvissuto', 'Deceduto'], 
                 filled=True,
                 rounded=True, 
                 fontsize=12,
                 impurity=False)

# Ciclo per pulire il testo
for n in nodi:
    testo = n.get_text().split('\n')
    # Pulizia: se c'è una condizione teniamo riga 0 e riga finale.
    # Se è una foglia, teniamo solo il risultato finale.
    if "<=" in testo[0]:
        n.set_text(f"{testo[0]}\n{testo[-1]}")
    else:
        n.set_text(testo[-1])

plt.title("L'Albero Decisionale: il 'Ragionamento' dell'IA", fontsize=20, pad=20)
plt.show()

In [ ]:
# Task di classificazione per prevedere il decesso ('DEATH_EVENT') tramite Random Forest
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Preparazione dei dati: dividiamo il dataset in set di Addestramento (80%) e Test (20%)
# Usiamo 'stratify=y' per assicurarci che la proporzione di decessi sia uguale in entrambi i set
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Inizializzazione e Addestramento del Modello
# La Random Forest crea una "foresta" di alberi decisionali per una predizione più robusta
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train, y_train.values.ravel()) # .values.ravel() evita errori di formato con Pandas

# 3. Predizione sui dati di Test (dati che il modello non ha mai visto)
y_pred_rf = rf_clf.predict(X_test)

# 4. Valutazione delle Performance
print("\n--- Risultati Random Forest ---")
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f"Accuratezza del Modello: {accuracy_rf:.2f}")

print("\nReport di Classificazione:")
print(classification_report(y_test, y_pred_rf))

# 5. Matrice di Confusione: quanti ne abbiamo indovinati e dove abbiamo sbagliato?
plt.figure(figsize=(8, 6))
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens')
plt.title('Matrice di Confusione - Random Forest')
plt.ylabel('Valore Reale (Sopravvissuto/Deceduto)')
plt.xlabel('Valore Predetto')
plt.show()

# 6. Analisi dell'Importanza delle Variabili (Feature Importance)
# Vediamo quali fattori hanno pesato di più per la decisione del modello
feature_importances = pd.DataFrame(
    rf_clf.feature_importances_,
    index = X_original.columns,
    columns=['importanza']
).sort_values('importanza', ascending=False)

print("\nImportanza delle variabili secondo la Random Forest:")
print(feature_importances)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# 1. Calcoliamo l'importanza per PREVEDERE I CLUSTER
rf_clusters = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clusters.fit(X_scaled, y_kmeans)
imp_clusters = pd.Series(rf_clusters.feature_importances_, index=X_original.columns).sort_values(ascending=False)

# 2. Calcoliamo l'importanza per PREVEDERE LA MORTE
rf_death = RandomForestClassifier(n_estimators=100, random_state=42)
rf_death.fit(X_scaled, y.values.ravel())
imp_death = pd.Series(rf_death.feature_importances_, index=X_original.columns).sort_values(ascending=False)

# 3. Visualizzazione di confronto
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), sharex=True)

# Plot Cluster Importance
sns.barplot(x=imp_clusters.values, y=imp_clusters.index, hue=imp_clusters.index, ax=ax1, palette="Blues_r", legend=False)
ax1.set_title("Cosa definisce i CLUSTER?", fontsize=13, fontweight='bold')

# Plot Death Importance
colors = ['red' if (idx == 'sex' or idx == 'smoking') else 'gray' for idx in imp_death.index]
sns.barplot(x=imp_death.values, y=imp_death.index, hue=imp_death.index, ax=ax2, palette=colors, legend=False)
ax2.set_title("Cosa definisce la MORTALITÀ?", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 🏁 Cosa abbiamo imparato?

Questo confronto finale svela il "trucco" dell'algoritmo e ci insegna una lezione fondamentale di medicina e data science.

<details>
<summary><b>🤔 1. Il Clustering è stato "distratto"?</b></summary>

Sì! Il clustering ha diviso i pazienti in base a **sesso e fumo** (le barre blu che abbiamo visto a sinistra) solo perché erano le differenze più facili da individuare matematicamente. 
Ha creato dei gruppi "sociali", ma non ci ha detto nulla sulla salute reale del cuore dei pazienti.
</details>

<br>

<details>
<summary><b>🩺 2. La Random Forest ha trovato la verità?</b></summary>

Esatto. Quando abbiamo dato al modello l'obiettivo di prevedere la mortalità, le variabili sesso e fumo sono "crollate" (barre rosse a destra). 
L'IA ha imparato che, per salvare una vita, bisogna ignorare il genere e guardare i dati vitali: **il cuore e i reni.**
</details>

<br>

<details>
<summary><b>💡 Conclusione: Il potere della Visualizzazione</b></summary>

Una visualizzazione chiara dei dati non serve solo a fare "bei grafici". Ci permette di capire quando un modello sta seguendo un **pregiudizio dei dati (bias)** invece di una reale utilità medica. 

**Il bravo Data Scientist è quello che sa mettere in dubbio i risultati della macchina!**
</details>

---

Created by Hari Calzi (MSc in CS at UniMi) for the DIGA Project – University of Milan

Contact: hari.calzi@studenti.unimi.it